# 19 — Stronger AHR Scout and Worker

Train a controlled pair on notebook 18's competitive, provenance-preserving teacher bank. The **96×6 control** starts from the previous best supervised model (step 11724); the **128×10 challenger** starts fresh. Both receive the same teacher examples and update allowance. These are different initialization strategies as well as different capacities, so this comparison does not isolate capacity alone.

**Inputs:** the complete `go9x9-competitive-v2` cache from notebook 18, frozen definitions from 00–03 and 12–13, and the existing step-11724 checkpoint for the control. **Outputs:** separate model/optimizer/scheduler/scaler/RNG/sampler checkpoints, development calibration and loss reports, and a candidate registry. **Expected runtime:** measured throughput determines duration; each candidate defaults to at most 20,000 updates or three active training hours. Step completion can occur much sooner. No claim of KataGo strength is made from supervised losses.

**Recovery:** checkpoints are written at every epoch/evaluation boundary and at least every 100 updates or 120 seconds. Each run retains the latest two valid states and the best development-loss state. Resume with `go2_k19_train(PROJECT, saved_config)` using the identical configuration; never change a run's data, sources, or initialization. `STOP` in its checkpoint directory stops after the current complete update. A detached tmux worker launches from the final Run All cell; its output includes attach and detach instructions. One writer owns each run.

Production training stays unstarted while this notebook is validated. Set `AHR_K19_MODE=validate` for the disposable CPU/GPU recovery checks. Dataset test payloads remain unopened. Notebook 20 decides promotion using fresh playing-strength games; notebook 19 never replaces the human-play champion automatically.

In [ ]:
from pathlib import Path
import os, json, nbformat
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
PROJECT=Path('/home/benjamin/workspace/ahr_go')
for cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(cell.source,globals())
for name in ('01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb',
             '12_teacher_dataset_and_targets.ipynb','13_teacher_pretraining_and_recovery.ipynb',
             '18_diagnosis_and_stronger_teacher_data.ipynb'):
    path=PROJECT/GO2_ROOT/name
    if path.exists():go2_import_exports(path,globals())

### Registered experiment and sources

The warm start copies verified model weights into a new optimizer run. It does not resume the old optimizer. A restart of an existing notebook-19 run restores its own exact optimizer and sampler state. Development losses select the saved `best`; this selection is separate from any playing champion.

In [ ]:
import dataclasses, copy, math, time, uuid, contextlib, shutil, subprocess, shlex
import numpy as np
import torch
import torch.nn.functional as F
GO2_K19_NOTEBOOK='19_stronger_scout_worker_training.ipynb'
GO2_K19_DATA_NOTEBOOK='18_diagnosis_and_stronger_teacher_data.ipynb'
GO2_K19_SCHEMA='ahr-k19-supervised-v1'

def go2_k19_sources(project=None):
    project=Path(project or PROJECT)
    names=('00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb',
           '02_scout_worker_network.ipynb','03_batched_puct_search.ipynb',
           '12_teacher_dataset_and_targets.ipynb','13_teacher_pretraining_and_recovery.ipynb',
           GO2_K19_DATA_NOTEBOOK,GO2_K19_NOTEBOOK)
    return {GO2_ROOT+'/'+name:go2_code_sha(project/GO2_ROOT/name) for name in names}

def go2_k19_config(run_id='ahr-k19-control-20261011',cache_id='go9x9-competitive-v2',device='cuda:1',variant='control'):
    if variant not in ('control','challenger'):raise ValueError('Choose control or challenger')
    initialization={'kind':'teacher_best','run_id':'ahr-v2-teacher-20261008','which':'best'} if variant=='control' else {'kind':'fresh'}
    return {'schema':GO2_K19_SCHEMA,'run_id':run_id,'cache_id':cache_id,'variant':variant,'device':device,
        'network':{'width':96,'blocks':6} if variant=='control' else {'width':128,'blocks':10},
        'initialization':initialization,'seed':26101119,'batch_size':512,'shuffle_block_rows':16384,
        'learning_rate':0.0003 if variant=='control' else 0.0005,'weight_decay':0.0001,
        'gradient_clip':2.0,'max_optimizer_steps':20000,'max_epochs':6,'time_budget_hours':3.0,
        'warmup_steps':200,'minimum_lr_fraction':0.05,'precision':'bfloat16' if str(device).startswith('cuda') else 'float32',
        'cpu_threads':4,'checkpoint_seconds':120.0,'checkpoint_optimizer_steps':100,
        'evaluate_every_steps':1000,'development_rows':8192,'train_rows_limit':None,
        'is_smoke':False,'fixture_data':False,'row_weight_scope':'policy',
        'loss_weights':{'policy':1.0,'teacher_value':1.0,'game_outcome':0.0,'score':0.5,'ownership':0.25,
                        'economic_policy':0.0,'economic_value':0.0},
        'target_claim':'masked searched teacher policy when valid; teacher estimates remain estimates',
        'selection_basis':'fixed_development_joint_loss_not_playing_strength'}

def go2_k19_validate_config(config):
    if not isinstance(config,dict):raise ValueError('Training configuration must be a mapping')
    base=go2_k19_config(variant=config.get('variant','control'))
    unknown=set(config)-set(base)
    if unknown:raise ValueError('Unknown notebook-19 settings: '+str(sorted(unknown)))
    result={**base,**copy.deepcopy(config)}
    for key in ('schema','target_claim','selection_basis'):
        if result[key]!=base[key]:raise ValueError('Unsupported contract: '+key)
    for key in ('run_id','cache_id'):
        if not result[key] or Path(result[key]).name!=result[key]:raise ValueError('Simple local identifier required: '+key)
    if result['device'] not in ('cpu','cuda:0','cuda:1'):raise ValueError('Use CPU or an explicit GPU')
    for key in ('batch_size','shuffle_block_rows','max_optimizer_steps','max_epochs','cpu_threads',
                'checkpoint_optimizer_steps','evaluate_every_steps','development_rows'):
        if type(result[key]) is not int or result[key]<1:raise ValueError('Positive integer required: '+key)
    for key in ('learning_rate','gradient_clip','checkpoint_seconds','time_budget_hours'):
        if not math.isfinite(result[key]) or result[key]<=0:raise ValueError('Positive finite setting required: '+key)
    if result['checkpoint_seconds']>900 or result['checkpoint_optimizer_steps']>250:raise ValueError('Checkpoint cadence exceeds agreement')
    if result['warmup_steps']<0 or not 0<result['minimum_lr_fraction']<=1:raise ValueError('Invalid learning-rate schedule')
    if result['precision'] not in ('float32','bfloat16') or (result['device']=='cpu' and result['precision']!='float32'):raise ValueError('Invalid precision/device')
    if result['train_rows_limit'] is not None and (type(result['train_rows_limit']) is not int or result['train_rows_limit']<1):raise ValueError('Invalid row limit')
    if result['row_weight_scope'] not in ('policy','all'):raise ValueError('Choose policy or all for row weight scope')
    if result['fixture_data'] and not result['is_smoke']:raise ValueError('Fixture data cannot masquerade as production')
    if set(result['loss_weights'])!=set(base['loss_weights']) or any(not math.isfinite(v) or v<0 for v in result['loss_weights'].values()):raise ValueError('Invalid loss weights')
    initialization=result['initialization']
    if initialization.get('kind') not in ('fresh','teacher_best','k19_checkpoint'):raise ValueError('Unsupported initialization')
    if initialization['kind']!='fresh' and not initialization.get('run_id'):raise ValueError('Initialization run ID required')
    Go2NetConfig(board_size=9,**result['network'])
    return result

def go2_k19_data_identity(dataset):
    return {'manifest_sha256':go2_digest(dataset['manifest']),'identity':dataset['identity'],
            'cache_path':str(dataset.get('cache_path','fixture')),
            'target_provenance':'teacher estimates and separately masked observed outcomes'}

def go2_k19_lr_factor(step,config):
    if config['warmup_steps'] and step<config['warmup_steps']:return (step+1)/config['warmup_steps']
    fraction=min(max((step-config['warmup_steps'])/max(1,config['max_optimizer_steps']-config['warmup_steps']),0.0),1.0)
    return config['minimum_lr_fraction']+(1-config['minimum_lr_fraction'])*0.5*(1+math.cos(math.pi*fraction))

def go2_k19_initial_player(project,config):
    init=config['initialization']
    if init['kind']=='fresh':return None,{'kind':'fresh','seed':config['seed']}
    if init['kind']=='teacher_best':
        model,meta=go2_load_teacher_player(project,init['run_id'],which=init.get('which','best'),device='cpu')
    else:
        model,meta=go2_k19_load_player(project,init['run_id'],which=init.get('which','best'),device='cpu')
    if dataclasses.asdict(model.config)!={'board_size':9,**config['network'],'scout_temperature':1.0,'worker_temperature':1.0,'rule_profile':GO2_RULE_PROFILE}:
        raise ValueError('Warm-start network must exactly match the registered architecture')
    pin={'kind':init['kind'],'run_id':meta['run_id'],'step':meta['global_step'],
         'checkpoint_file':meta['checkpoint_file'],'checkpoint_sha256':meta['checkpoint_sha256'],
         'model_identity':meta['model_identity'],'optimizer_copied':False}
    return copy.deepcopy(model.state_dict()),pin

### Weighted Scout, Worker and auxiliary objectives

Every policy remains a full legal distribution. Row weights change the emphasis of examples, never the definition of win/loss. Score and ownership labels use their own validity masks. Optional notebook-20 economics targets require explicit weights and valid normalized targets; their default weight here is zero.

In [ ]:
def go2_k19_validate_batch(batch):
    go2_teacher_validate_batch(batch)
    n=len(batch['value']);device=batch['value'].device
    own=batch['ownership'].float().reshape(n,81);own_valid=batch['ownership_valid'].bool()
    if (own[own_valid].abs()>1.00001).any():raise ValueError('Valid ownership targets must lie in [-1,1]')
    if 'row_weight' in batch:
        weights=batch['row_weight'].float()
        if weights.shape!=(n,) or not torch.isfinite(weights).all() or (weights<=0).any():raise ValueError('Row weights must be finite and positive')
    for field in ('policy_valid','value_valid','economic_value_valid','economic_policy_valid'):
        if field in batch and tuple(batch[field].shape)!=(n,):raise ValueError('One validity flag per row required: '+field)
    if 'economic_value' in batch:
        values=batch['economic_value'].float()
        if values.shape!=(n,) or not torch.isfinite(values).all() or (values.abs()>1.00001).any():raise ValueError('Economic Worker target must be a finite signed value in [-1,1]')
    if 'economic_policy' in batch:
        policy=batch['economic_policy'].float()
        if policy.shape!=(n,82) or not torch.isfinite(policy).all() or (policy<0).any() or (policy.masked_select(~batch['legal'].bool()).abs()>1e-6).any() or not torch.allclose(policy.sum(1),torch.ones(n,device=device),atol=.002,rtol=0):raise ValueError('Invalid economic policy distribution')
    return batch

def go2_k19_weighted_mean(values,weights,mask=None):
    if mask is not None:weights=weights*mask.bool().float()
    denominator=weights.sum()
    return (values*weights).sum()/denominator.clamp_min(1e-12) if bool(denominator>0) else values.sum()*0.0

def go2_k19_d4(batch,symmetries):
    result=go2_teacher_d4(batch,symmetries)
    if 'economic_policy' in batch:
        helper=dict(batch);helper['policy']=batch['economic_policy']
        result['economic_policy']=go2_teacher_d4(helper,symmetries)['policy']
    return result

def go2_k19_loss(model,batch,config):
    go2_k19_validate_batch(batch)
    output=model(batch['features'])
    logits=(-output['scout_energy'].float()/model.config.scout_temperature).masked_fill(~batch['legal'].bool(),-1e9)
    policy_log=F.log_softmax(logits,dim=1)
    value_log=F.log_softmax(-output['worker_energy'].float()/model.config.worker_temperature,dim=1)
    value=batch['value'].float().clamp(-1,1)
    soft_value=torch.stack(((1-value)/2,torch.zeros_like(value),(1+value)/2),dim=1)
    weights=batch.get('row_weight',torch.ones_like(value)).float()
    auxiliary_weights=weights if config.get('row_weight_scope','policy')=='all' else torch.ones_like(weights)
    components={
        'policy':go2_k19_weighted_mean(-(batch['policy'].float()*policy_log).sum(1),weights,batch.get('policy_valid')),
        'teacher_value':go2_k19_weighted_mean(-(soft_value*value_log).sum(1),auxiliary_weights,batch.get('value_valid')),
        'game_outcome':go2_k19_weighted_mean(-(batch['outcome'].float()*value_log).sum(1),auxiliary_weights,batch['outcome_valid']),
        'score':go2_k19_weighted_mean((output['score'].float()-batch['score'].float()).square(),auxiliary_weights,batch['score_valid']),
        'ownership':go2_k19_weighted_mean((output['ownership'].float()-batch['ownership'].float().reshape(-1,81)).square().mean(1),auxiliary_weights,batch['ownership_valid']),
        'economic_policy':output['scout_energy'].sum()*0.0,
        'economic_value':output['worker_energy'].sum()*0.0}
    if config['loss_weights']['economic_policy']:
        if 'economic_policy' not in batch or 'economic_policy_valid' not in batch:raise ValueError('Positive economic policy weight requires explicit target and validity')
        components['economic_policy']=go2_k19_weighted_mean(-(batch['economic_policy'].float()*policy_log).sum(1),weights,batch['economic_policy_valid'])
    if config['loss_weights']['economic_value']:
        if 'economic_value' not in batch or 'economic_value_valid' not in batch:raise ValueError('Positive economic value weight requires explicit target and validity')
        economic=batch['economic_value'].float();target=torch.stack(((1-economic)/2,torch.zeros_like(economic),(1+economic)/2),dim=1)
        components['economic_value']=go2_k19_weighted_mean(-(target*value_log).sum(1),auxiliary_weights,batch['economic_value_valid'])
    loss=sum(config['loss_weights'][key]*item for key,item in components.items())
    if not torch.isfinite(loss):raise RuntimeError('Non-finite supervised objective')
    return {'loss':loss,'components':components,'output':output,'policy_log':policy_log,'value_log':value_log}

### Development calibration

Development examples and indices are fixed before training. Reports retain sample counts for competitive values, phases and strong/weak opponent strata. Teacher calibration compares predictions with teacher probabilities; observed calibration uses only compatible completed-game outcomes. Neither is a playing-strength result.

In [ ]:
def go2_k19_metric_evaluate(model,dataset,indices,config,sample_fn=None):
    sample_fn=sample_fn or go2_k19_sample_batch
    was_training=model.training;model.eval();groups={};bins=[{'rows':0,'prediction_sum':0.,'teacher_sum':0.,'observed_rows':0,'observed_prediction_sum':0.,'observed_win_sum':0.} for _ in range(10)]
    objective=0.;objective_rows=0
    try:
        with torch.inference_mode():
            for offset in range(0,len(indices),config['batch_size']):
                ids=indices[offset:offset+config['batch_size']]
                batch=sample_fn(dataset,ids,device=config['device'],split='dev');result=go2_k19_loss(model,batch,config)
                n=len(ids);objective+=float(result['loss'].cpu())*n;objective_rows+=n
                policy=batch['policy'].float();pred=result['policy_log'].exp();probs=result['value_log'].exp()
                pwin=probs[:,2];teacher=(1+batch['value'].float())/2
                kl=(policy*(policy.clamp_min(1e-12).log()-result['policy_log'])).sum(1)
                value_error=(probs[:,2]-probs[:,0]-batch['value']).square()
                ply=batch.get('ply',torch.zeros(n,device=pwin.device,dtype=torch.long))
                opponent=batch.get('opponent',torch.full((n,),-1,device=pwin.device,dtype=torch.long))
                masks={'all':torch.ones(n,device=pwin.device,dtype=torch.bool),
                    'competitive_abs_value_below_095':batch['value'].abs()<.95,
                    'close_abs_value_below_080':batch['value'].abs()<.8,
                    'opening_ply_below_20':ply<20,'middle_ply_20_to_59':(ply>=20)&(ply<60),
                    'late_ply_at_least_60':ply>=60,'weak_opponent_strata_0_to_3':(opponent>=0)&(opponent<4),
                    'strong_opponent_strata_4_to_7':opponent>=4}
                for name,mask in masks.items():
                    count=int(mask.sum())
                    if not count:continue
                    row=groups.setdefault(name,{'rows':0,'policy_kl_sum':0.,'policy_agreement_sum':0.,'value_mse_sum':0.,'teacher_brier_sum':0.,
                        'score_rows':0,'score_mae_points_sum':0.,'observed_rows':0,'observed_brier_sum':0.,'observed_accuracy_sum':0.,
                        'ownership_rows':0,'searched_policy_rows':0,'removed_policy_mass_sum':0.})
                    row['rows']+=count;row['policy_kl_sum']+=float(kl[mask].sum().cpu());row['policy_agreement_sum']+=float((pred.argmax(1)[mask]==policy.argmax(1)[mask]).float().sum().cpu())
                    row['value_mse_sum']+=float(value_error[mask].sum().cpu());row['teacher_brier_sum']+=float((pwin[mask]-teacher[mask]).square().sum().cpu())
                    valid=mask&batch['score_valid'].bool();row['score_rows']+=int(valid.sum())
                    row['score_mae_points_sum']+=float((result['output']['score'].float()[valid]-batch['score'].float()[valid]).abs().sum().cpu())*88.5
                    valid=mask&batch['outcome_valid'].bool();row['observed_rows']+=int(valid.sum())
                    row['observed_brier_sum']+=float((probs[valid]-batch['outcome'][valid]).square().sum().cpu())
                    row['observed_accuracy_sum']+=float((probs[valid].argmax(1)==batch['outcome'][valid].argmax(1)).float().sum().cpu())
                    row['ownership_rows']+=int((mask&batch['ownership_valid'].bool()).sum())
                    row['searched_policy_rows']+=int((mask&batch.get('policy_searched',torch.zeros_like(mask)).bool()).sum())
                    row['removed_policy_mass_sum']+=float(batch.get('removed_policy_mass',torch.zeros_like(pwin))[mask].sum().cpu())
                ids_bin=(pwin*10).long().clamp(0,9)
                for b in range(10):
                    mask=ids_bin==b;row=bins[b];row['rows']+=int(mask.sum());row['prediction_sum']+=float(pwin[mask].sum().cpu());row['teacher_sum']+=float(teacher[mask].sum().cpu())
                    valid=mask&batch['outcome_valid'].bool();row['observed_rows']+=int(valid.sum());row['observed_prediction_sum']+=float(pwin[valid].sum().cpu());row['observed_win_sum']+=float(batch['outcome'][valid,2].sum().cpu())
    finally:model.train(was_training)
    if not objective_rows:raise RuntimeError('Empty development bank')
    results={}
    for name,row in groups.items():
        n=row['rows'];nv=row['observed_rows'];ns=row['score_rows']
        results[name]={'rows':n,'policy_kl':row['policy_kl_sum']/n,'policy_top1_agreement':row['policy_agreement_sum']/n,
            'teacher_value_mse':row['value_mse_sum']/n,'teacher_win_probability_brier':row['teacher_brier_sum']/n,
            'teacher_score_rows':ns,'teacher_score_mae_points':row['score_mae_points_sum']/ns if ns else None,
            'observed_outcome_rows':nv,'observed_outcome_brier':row['observed_brier_sum']/nv if nv else None,
            'observed_outcome_accuracy':row['observed_accuracy_sum']/nv if nv else None,
            'ownership_rows':row['ownership_rows'],'ownership_valid_fraction':row['ownership_rows']/n,
            'searched_policy_rows':row['searched_policy_rows'],'removed_policy_mass_mean':row['removed_policy_mass_sum']/n}
    calibration=[]
    for b,row in enumerate(bins):
        n=row['rows'];nv=row['observed_rows']
        calibration.append({'bin_low':b/10,'bin_high':(b+1)/10,'rows':n,
            'predicted_win_mean':row['prediction_sum']/n if n else None,'teacher_win_mean':row['teacher_sum']/n if n else None,
            'observed_rows':nv,'observed_predicted_win_mean':row['observed_prediction_sum']/nv if nv else None,
            'observed_win_fraction':row['observed_win_sum']/nv if nv else None})
    return {'joint_loss':objective/objective_rows,'selection_score':objective/objective_rows,
            'selection_basis':config['selection_basis'],'rows':objective_rows,'groups':results,'calibration':calibration,
            'is_smoke':config['is_smoke'],'claim':'development predictive evidence, not Elo or proved optimality'}

### Data dispatch and resumable updates

The disposable recovery bank contains explicitly synthetic targets on legal causal states. It checks optimizer, masks, sampler and RNG recovery; its losses measure engineering correctness only. Production data comes exclusively from notebook 18's immutable cache.

In [ ]:
def go2_k19_fixture_dataset(project=None):
    states=[];state=go2_initial_state(9,7.5)
    for action in (40,30,41,31,50,49,42,32,51,48,39,29):
        states.append(state);state=go2_step(state,action)
    states.extend((state,go2_initial_state(9,7.5)))
    arrays={};features=[];legal=[];policies=[]
    for i in range(64):
        s=states[i%len(states)];features.append(go2_encode_state(s));mask=np.asarray(go2_legal_mask(s),dtype=np.bool_);legal.append(mask)
        policy=np.zeros(82,np.float32);eligible=np.flatnonzero(mask);policy[eligible]=np.arange(1,len(eligible)+1,dtype=np.float32);policy/=policy.sum();policies.append(policy)
    value=np.sin(np.arange(64,dtype=np.float32)*.7)*.8
    arrays.update(features=np.asarray(features,np.float32),legal=np.asarray(legal),policy=np.asarray(policies),value=value,
        score=value*.2,score_valid=np.arange(64)%3!=0,outcome=np.stack((value<0,np.zeros(64,dtype=np.bool_),value>=0),1).astype(np.float32),
        outcome_valid=np.arange(64)%4==0,ownership=np.zeros((64,9,9),np.float32),ownership_valid=np.arange(64)%5==0,
        row_weight=1+(np.arange(64,dtype=np.float32)%3),policy_searched=np.arange(64)%2==0,teacher_turn=np.arange(64)%2==0,
        opponent=np.arange(64,dtype=np.int64)%8,ply=np.arange(64,dtype=np.int64)*2,removed_policy_mass=np.zeros(64,np.float32))
    digest=hashlib.sha256()
    for key,value0 in sorted(arrays.items()):digest.update(key.encode());digest.update(value0.tobytes())
    manifest={'schema':'disposable-k19-legal-state-fixture-v1','counts':{'train':40,'dev':24},'synthetic_targets':True,'test_consumed':False,'arrays_sha256':digest.hexdigest()}
    return {'manifest':manifest,'identity':go2_digest(manifest),'cache_path':'disposable-k19-legal-state-fixture-v1',
            'train':{'count':40,'row_count':40},'dev':{'count':24,'row_count':24},'_fixture':arrays}

def go2_k19_open_data(project,config):
    if config['fixture_data']:return go2_k19_fixture_dataset(project)
    if 'go2_k18_dataset' not in globals():raise RuntimeError('Run notebook 18 first; its exported data loader is unavailable')
    return go2_k18_dataset(project,cache_id=config['cache_id'],verify=True)

def go2_k19_sample_batch(dataset,indices,device='cpu',split='train'):
    if '_fixture' not in dataset:return go2_k18_sample_batch(dataset,indices,device=device,split=split)
    if split not in ('train','dev'):raise ValueError('Only development/training fixture views are allowed')
    ids=np.asarray(indices.detach().cpu() if isinstance(indices,torch.Tensor) else indices,dtype=np.int64)
    if ids.ndim!=1 or not len(ids) or (ids<0).any() or (ids>=dataset[split]['count']).any():raise ValueError('Invalid fixture batch cursor')
    if split=='dev':ids=ids+40
    flags={'legal','score_valid','outcome_valid','ownership_valid','policy_searched','teacher_turn'}
    return {name:torch.as_tensor(value[ids],device=device,dtype=torch.bool if name in flags else torch.long if name in ('opponent','ply') else torch.float32) for name,value in dataset['_fixture'].items()}

def go2_k19_progress(state,entry,status):
    return {'schema':GO2_K19_SCHEMA,'run_id':state['config']['run_id'],'variant':state['config']['variant'],
        'status':status,'updated_at_utc':go2_now(),'global_step':state['global_step'],'optimizer_step':state['global_step'],
        'epoch':state['sampler']['epoch'],'epoch_cursor':state['sampler']['cursor'],'training_rows':state['sampler']['train_rows'],
        'rows_seen':state['sampler']['rows_seen'],'active_training_seconds':state['active_seconds'],
        'time_budget_hours':state['config']['time_budget_hours'],'checkpoint':entry,
        'last_loss':state['metrics']['last_loss'],'last_development':state['metrics']['last_development'],
        'best_development_joint_loss':state['metrics']['best_development_joint_loss'],
        'initialization':state['identity']['initialization'],'data_identity':state['identity']['data'],
        'is_smoke':state['config']['is_smoke'],'selection_basis':state['config']['selection_basis']}

def go2_k19_train(project=None,config=None,max_steps=None,on_checkpoint=None):
    project=go2_project(project);config=go2_k19_validate_config(config or go2_k19_config())
    if max_steps is not None and (type(max_steps) is not int or max_steps<1):raise ValueError('max_steps is an additional positive update allowance')
    go2_teacher_determinism(config)
    dataset=go2_k19_open_data(project,config);train_rows=int(dataset['train']['count']);dev_rows=int(dataset['dev']['count'])
    if config['train_rows_limit'] is not None:train_rows=min(train_rows,config['train_rows_limit'])
    if min(train_rows,dev_rows)<1:raise RuntimeError('Both training and development banks are required')
    initial_weights,initial_pin=go2_k19_initial_player(project,config)
    identity={'schema':GO2_SCHEMA,'training_schema':GO2_K19_SCHEMA,'sources':go2_k19_sources(project),
        'config':copy.deepcopy(config),'data':go2_k19_data_identity(dataset),'initialization':initial_pin,
        'rules':GO2_RULE_PROFILE,'torch':torch.__version__,'numpy':np.__version__}
    run_dir=project/'checkpoints'/config['run_id'];report_dir=project/'reports/ahr_v2'/config['run_id'];report_dir.mkdir(parents=True,exist_ok=True)
    with go2_checkpoint_lock(run_dir):
        saved,entry=go2_load_checkpoint(run_dir,identity)
        go2_seed(config['seed'])
        model=Go2Network(Go2NetConfig(board_size=9,**config['network'])).to(config['device'])
        if initial_weights is not None:model.load_state_dict(initial_weights)
        optimizer=torch.optim.AdamW(model.parameters(),lr=config['learning_rate'],weight_decay=config['weight_decay'])
        scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:go2_k19_lr_factor(step,config))
        scaler=torch.amp.GradScaler('cuda',enabled=False)
        sampler=torch.Generator(device='cpu').manual_seed(config['seed']+101)
        dev_generator=torch.Generator(device='cpu').manual_seed(config['seed']+303)
        development_indices=torch.randperm(dev_rows,generator=dev_generator)[:min(config['development_rows'],dev_rows)]
        state={'schema':GO2_SCHEMA,'identity':identity,'config':config,'global_step':0,'stage':'competitive_teacher_training',
            'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':{},'active_seconds':0.,
            'sampler':{'epoch':0,'cursor':0,'rows_seen':0,'train_rows':train_rows,
                       'permutation':go2_teacher_permutation(train_rows,sampler,config['shuffle_block_rows'])},
            'development_indices':development_indices,'data_identity':identity['data'],'adapters':{},'replay':{},
            'metrics':{'optimizer_step':0,'last_loss':None,'loss_ema':None,'best_development_joint_loss':None,
                'last_development':None,'last_evaluation_step':-1,'recent_losses':[],'development_history':[],
                'target_rows_seen':{'policy':0,'teacher_value':0,'score':0,'ownership':0,'observed_outcome':0,'searched_policy':0}}}
        if saved is not None:
            state=saved;model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
            go2_restore_rng(state['rng'],{'sampler':sampler});go2_invalidate_model_identity(model)
            go2_k19_reconcile_logs(report_dir,state['global_step'])
            if state['sampler']['train_rows']!=train_rows or len(state['sampler']['permutation'])!=train_rows:raise RuntimeError('Saved sampler bank extent changed')
        go2_atomic_json(run_dir/'config.json',config)
        last_saved=-1 if entry is None else state['global_step'];last_checkpoint=time.monotonic();steps_saved=state['global_step'];processed=0;status='RUNNING';pending_best=False
        def progress():
            value=go2_k19_progress(state,entry,status);go2_atomic_json(run_dir/'progress.json',value);go2_atomic_json(report_dir/'status.json',value)
        def snapshot(force=False):
            nonlocal entry,last_saved,last_checkpoint,steps_saved,pending_best
            if state['global_step']==last_saved:return
            if not force and time.monotonic()-last_checkpoint<config['checkpoint_seconds'] and state['global_step']-steps_saved<config['checkpoint_optimizer_steps']:return
            state.update(model=copy.deepcopy(model.state_dict()),optimizer=copy.deepcopy(optimizer.state_dict()),scheduler=copy.deepcopy(scheduler.state_dict()),
                scaler=copy.deepcopy(scaler.state_dict()),rng=go2_capture_rng(cuda_devices=go2_teacher_cuda_devices(config),generators={'sampler':sampler}))
            entry=go2_save_checkpoint(run_dir,state,best=pending_best,resume_cell='19: go2_k19_train(PROJECT,json.loads((PROJECT/"checkpoints"/run_id/"config.json").read_text()))')
            pending_best=False;last_saved=state['global_step'];last_checkpoint=time.monotonic();steps_saved=state['global_step']
            go2_k19_register_candidate(project,state)
            if on_checkpoint is not None:
                before=go2_capture_rng(cuda_devices=go2_teacher_cuda_devices(config),generators={'sampler':sampler})
                try:on_checkpoint(copy.deepcopy(state),copy.deepcopy(entry))
                finally:go2_restore_rng(before,{'sampler':sampler})
        def evaluate():
            nonlocal pending_best
            before=go2_capture_rng(cuda_devices=go2_teacher_cuda_devices(config),generators={'sampler':sampler});started=time.monotonic()
            try:audit=go2_k19_metric_evaluate(model,dataset,state['development_indices'],config)
            finally:go2_restore_rng(before,{'sampler':sampler})
            audit.update(optimizer_step=state['global_step'],duration_seconds=time.monotonic()-started)
            metrics=state['metrics'];metrics['last_development']=audit;metrics['last_evaluation_step']=state['global_step'];metrics['development_history']=(metrics['development_history']+[audit])[-128:]
            old=metrics['best_development_joint_loss']
            if old is None or audit['selection_score']<old:metrics['best_development_joint_loss']=audit['selection_score'];pending_best=True
            with (report_dir/'development.jsonl').open('a') as handle:handle.write(json.dumps(audit)+'\n');handle.flush()
        if saved is None:evaluate()
        snapshot(True);progress()
        try:
            while True:
                if (run_dir/'STOP').exists():status='STOPPED';break
                if state['active_seconds']>=config['time_budget_hours']*3600:status='TIME_BUDGET_COMPLETE';break
                if state['global_step']>=config['max_optimizer_steps']:status='STEP_LIMIT_COMPLETE';break
                if state['sampler']['epoch']+int(state['sampler']['cursor']==train_rows)>=config['max_epochs']:status='EPOCH_LIMIT_COMPLETE';break
                if max_steps is not None and processed>=max_steps:status='PAUSED';break
                started=time.monotonic();sam=state['sampler']
                if sam['cursor']==train_rows:sam['epoch']+=1;sam['cursor']=0;sam['permutation']=go2_teacher_permutation(train_rows,sampler,config['shuffle_block_rows'])
                first=sam['cursor'];last=min(first+config['batch_size'],train_rows);ids=sam['permutation'][first:last]
                batch=go2_k19_sample_batch(dataset,ids,device=config['device'],split='train');go2_k19_validate_batch(batch)
                symmetries=torch.randint(0,8,(len(ids),),generator=sampler);batch=go2_k19_d4(batch,symmetries)
                model.train();optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda' if config['device'].startswith('cuda') else 'cpu',dtype=torch.bfloat16,enabled=config['precision']=='bfloat16'):
                    result=go2_k19_loss(model,batch,config)
                result['loss'].backward();gradient_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),config['gradient_clip'],error_if_nonfinite=True)
                optimizer.step();scheduler.step();go2_invalidate_model_identity(model)
                sam['cursor']=last;sam['rows_seen']+=len(ids);state['global_step']+=1;processed+=1
                metrics=state['metrics'];metrics['optimizer_step']=state['global_step'];metrics['last_loss']=float(result['loss'].detach().cpu());metrics['loss_ema']=metrics['last_loss'] if metrics['loss_ema'] is None else .95*metrics['loss_ema']+.05*metrics['last_loss']
                target_rows={'policy':len(ids),'teacher_value':len(ids),'score':int(batch['score_valid'].sum()),'ownership':int(batch['ownership_valid'].sum()),
                    'observed_outcome':int(batch['outcome_valid'].sum()),'searched_policy':int(batch.get('policy_searched',torch.zeros(len(ids),dtype=torch.bool)).sum())}
                record={'optimizer_step':state['global_step'],'rows':len(ids),'loss':metrics['last_loss'],'gradient_norm':float(gradient_norm.cpu()),
                    'components':{key:float(value.detach().cpu()) for key,value in result['components'].items()},'learning_rate':optimizer.param_groups[0]['lr'],'target_rows':target_rows}
                for key,count in target_rows.items():metrics['target_rows_seen'][key]+=count
                metrics['recent_losses']=(metrics['recent_losses']+[record])[-256:]
                state['active_seconds']+=time.monotonic()-started
                with (report_dir/'metrics.jsonl').open('a') as handle:handle.write(json.dumps(record)+'\n');handle.flush()
                boundary=sam['cursor']==train_rows
                if state['global_step']-metrics['last_evaluation_step']>=config['evaluate_every_steps'] or boundary:evaluate()
                snapshot(boundary or state['metrics']['last_evaluation_step']==state['global_step'])
                if state['global_step']%20==0 or boundary or state['metrics']['last_evaluation_step']==state['global_step']:progress()
        except BaseException as error:
            valid,valid_entry=go2_load_checkpoint(run_dir,identity)
            failed=go2_k19_progress(valid,valid_entry,'ERROR') if valid is not None else {'run_id':config['run_id'],'status':'ERROR'}
            failed.update(error=repr(error),discarded_uncommitted_update=True,updated_at_utc=go2_now());go2_atomic_json(report_dir/'status.json',failed);go2_atomic_json(run_dir/'progress.json',failed)
            raise
        if status in ('STEP_LIMIT_COMPLETE','EPOCH_LIMIT_COMPLETE','TIME_BUDGET_COMPLETE') and state['global_step']!=state['metrics']['last_evaluation_step'] and state['global_step']!=last_saved:evaluate()
        snapshot(True);progress()
        state['status']=status
        return state

def go2_k19_load_player(project=None,run_id=None,which='best',device='cpu'):
    project=go2_project(project)
    if run_id is None:raise ValueError('An explicit notebook-19 candidate run ID is required; no champion pointer is changed')
    state,entry=go2_load_checkpoint(project/'checkpoints'/run_id,which=which)
    if state is None:raise FileNotFoundError('Notebook-19 checkpoint missing: '+run_id)
    if state['identity'].get('training_schema')!=GO2_K19_SCHEMA or state['identity']['sources']!=go2_k19_sources(project):raise RuntimeError('Notebook-19 checkpoint schema or source identity changed')
    with torch.random.fork_rng(devices=[]):model=Go2Network(Go2NetConfig(board_size=9,**state['config']['network']))
    model.load_state_dict(state['model']);model.to(device);model.eval();model.requires_grad_(False)
    metadata={'run_id':run_id,'global_step':state['global_step'],'optimizer_step':state['global_step'],'model_identity':go2_model_identity(model),
        'checkpoint_file':entry['file'],'checkpoint_sha256':entry['sha256'],'config':state['config'],'network':state['config']['network'],
        'variant':state['config']['variant'],'sources':state['identity']['sources'],'data_identity':state['identity']['data'],
        'initialization':state['identity']['initialization'],'is_smoke':state['config']['is_smoke'],'development':state['metrics']['last_development'],
        'selection_basis':state['config']['selection_basis'],'target_rows_seen':state['metrics']['target_rows_seen'],'training_source':'competitive_teacher_estimates_not_game_proofs'}
    return model,metadata

In [ ]:
def go2_k19_reconcile_logs(report_dir,saved_step):
    for name in ('metrics.jsonl','development.jsonl'):
        path=Path(report_dir)/name
        if not path.exists():continue
        records=[]
        for line in path.read_text().splitlines():
            if not line.strip():continue
            try:record=json.loads(line)
            except json.JSONDecodeError:raise RuntimeError('Invalid persisted training log: '+str(path))
            if int(record['optimizer_step'])<=saved_step:records.append(record)
        go2_atomic_bytes(path,('\n'.join(json.dumps(row) for row in records)+('\n' if records else '')).encode())

def go2_k19_register_candidate(project,state):
    run_dir=Path(project)/'checkpoints'/state['config']['run_id'];manifest=json.loads((run_dir/'manifest.json').read_text())
    result={'schema':'ahr-k19-candidate-registry-v1','run_id':state['config']['run_id'],'variant':state['config']['variant'],
        'latest':manifest['latest'],'best_development_loss':manifest['best'],'manifest_sha256':go2_file_sha(run_dir/'manifest.json'),
        'global_step':state['global_step'],'sources':state['identity']['sources'],'data_identity':state['identity']['data'],
        'is_smoke':state['config']['is_smoke'],'champion_selected':False,'selection_basis':state['config']['selection_basis']}
    go2_atomic_json(Path(project)/'reports/ahr_v2'/state['config']['run_id']/'candidate_registry.json',result)
    return result

def go2_k19_scientific_state(state):
    result={key:copy.deepcopy(state[key]) for key in ('model','optimizer','scheduler','scaler','rng','sampler','global_step','development_indices','metrics','adapters','replay')}
    for audit in result['metrics']['development_history']:audit.pop('duration_seconds',None)
    if result['metrics']['last_development'] is not None:result['metrics']['last_development'].pop('duration_seconds',None)
    return result

def go2_k19_bootstrap_text(project):
    return "from pathlib import Path\nimport os,nbformat,json\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\nPROJECT=Path("+repr(str(project))+")\nfor cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:\n    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())\nfor name in ('01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb','12_teacher_dataset_and_targets.ipynb','13_teacher_pretraining_and_recovery.ipynb',GO2_K19_DATA_NOTEBOOK,GO2_K19_NOTEBOOK):\n    go2_import_exports(PROJECT/GO2_ROOT/name,globals())".replace('GO2_K19_DATA_NOTEBOOK',repr(GO2_K19_DATA_NOTEBOOK)).replace('GO2_K19_NOTEBOOK',repr(GO2_K19_NOTEBOOK))

def go2_k19_fresh_fixture_worker(project,config,steps,label):
    from nbclient import NotebookClient
    import nbformat
    folder=Path(project)/'reports/ahr_v2'/config['run_id'];folder.mkdir(parents=True,exist_ok=True);path=folder/(label+'.ipynb')
    run="CONFIG=json.loads("+repr(json.dumps(config))+")\nSTATE=go2_k19_train(PROJECT,CONFIG,max_steps="+str(int(steps))+")\nprint({'step':STATE['global_step'],'cursor':STATE['sampler']['cursor'],'status':STATE['status']})"
    worker=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('Disposable optimizer/recovery fixture; synthetic labels, not a strength result.'),
        nbformat.v4.new_code_cell(go2_k19_bootstrap_text(project)),nbformat.v4.new_code_cell(run)],
        metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(worker,path);NotebookClient(worker,timeout=240,kernel_name='ahr-go',resources={'metadata':{'path':str(project)}}).execute();nbformat.write(worker,path)
    state,_=go2_load_checkpoint(Path(project)/'checkpoints'/config['run_id'])
    return state

def go2_k19_resume_fixture(project,device):
    token=uuid.uuid4().hex[:10];config=go2_k19_config('k19-fixture-'+token+'-continuous',device=device,variant='challenger')
    config.update(network={'width':8,'blocks':1},initialization={'kind':'fresh'},batch_size=8,max_optimizer_steps=6,max_epochs=3,
        time_budget_hours=.05,warmup_steps=0,development_rows=16,evaluate_every_steps=2,checkpoint_optimizer_steps=2,
        is_smoke=True,fixture_data=True,shuffle_block_rows=16)
    full=go2_k19_fresh_fixture_worker(project,config,6,'continuous')
    resumed_config={**copy.deepcopy(config),'run_id':'k19-fixture-'+token+'-resumed'}
    interrupted=go2_k19_fresh_fixture_worker(project,resumed_config,3,'interrupted')
    if interrupted['global_step']!=3 or interrupted['sampler']['cursor']!=24:raise AssertionError('Fixture interruption did not preserve a partial epoch cursor')
    resumed=go2_k19_fresh_fixture_worker(project,resumed_config,3,'resumed')
    if not go2_nested_equal(go2_k19_scientific_state(full),go2_k19_scientific_state(resumed)):raise AssertionError('Exact optimizer/sampler recovery differs: '+device)
    run_dir=Path(project)/'checkpoints'/resumed_config['run_id'];manifest,states=go2_manifest_states(run_dir)
    if len(states)<2 or manifest['best'] not in states:raise AssertionError('Latest-two/best retention failed')
    model,meta=go2_k19_load_player(project,resumed_config['run_id'],which='latest',device='cpu')
    if go2_tensor_digest(model.state_dict())!=go2_tensor_digest(resumed['model']):raise AssertionError('Frozen player loading changed weights')
    wrong=copy.deepcopy(resumed['identity']);wrong['data']['manifest_sha256']='0'*64
    try:go2_load_checkpoint(run_dir,wrong)
    except RuntimeError:pass
    else:raise AssertionError('Data identity mismatch was silently accepted')
    return {'device':device,'precision':config['precision'],'steps':resumed['global_step'],'rows_seen':resumed['sampler']['rows_seen'],
        'epoch':resumed['sampler']['epoch'],'cursor':resumed['sampler']['cursor'],'retained_checkpoints':len(states),
        'model_sha256':go2_tensor_digest(resumed['model']),'exact_scientific_state':True,'fresh_kernel_per_invocation':True,
        'data_mismatch_refused':True,'continuous_run_id':config['run_id'],'resumed_run_id':resumed_config['run_id']}

def go2_k19_validate(project=None,require_gpu=True):
    project=go2_project(project);dataset=go2_k19_fixture_dataset(project);batch=go2_k19_sample_batch(dataset,np.arange(8),device='cpu')
    go2_k19_validate_batch(batch);config=go2_k19_config(device='cpu',variant='challenger');config.update(network={'width':8,'blocks':1},initialization={'kind':'fresh'})
    with torch.random.fork_rng(devices=[]):model=Go2Network(Go2NetConfig(width=8,blocks=1))
    baseline=go2_k19_loss(model,batch,config)
    masked=copy.deepcopy(batch);masked['score_valid']=torch.zeros_like(masked['score_valid']);masked['ownership_valid']=torch.zeros_like(masked['ownership_valid'])
    check=go2_k19_loss(model,masked,config)
    if float(check['components']['score'])!=0 or float(check['components']['ownership'])!=0:raise AssertionError('Invalid auxiliary labels contributed loss')
    nonuniform=copy.deepcopy(batch);nonuniform['row_weight']=torch.arange(1,9,dtype=torch.float32)
    weighted=go2_k19_loss(model,nonuniform,config)
    if not torch.equal(weighted['components']['teacher_value'],baseline['components']['teacher_value']):raise AssertionError('Policy regret weights changed the Worker target objective')
    if torch.equal(weighted['components']['policy'],baseline['components']['policy']):raise AssertionError('Policy row weights have no effect')
    bad=copy.deepcopy(batch);bad['row_weight'][0]=float('nan')
    try:go2_k19_loss(model,bad,config)
    except ValueError:pass
    else:raise AssertionError('Invalid row weight accepted')
    economics=copy.deepcopy(config);economics['loss_weights']['economic_value']=.1
    try:go2_k19_loss(model,batch,economics)
    except ValueError:pass
    else:raise AssertionError('Missing economic target accepted')
    fixtures=[go2_k19_resume_fixture(project,'cpu')]
    if torch.cuda.is_available() and torch.cuda.device_count()>=2:fixtures.append(go2_k19_resume_fixture(project,'cuda:1'))
    elif require_gpu:raise RuntimeError('GPU1 exact recovery is required before the production GPU launch')
    import tempfile
    source=project/'checkpoints'/fixtures[0]['resumed_run_id']
    with tempfile.TemporaryDirectory(prefix='k19-corrupt-',dir=project/'checkpoints') as directory:
        bad_dir=Path(directory);shutil.copytree(source,bad_dir,dirs_exist_ok=True);manifest=json.loads((bad_dir/'manifest.json').read_text());payload=bad_dir/manifest['latest'];raw=bytearray(payload.read_bytes());raw[len(raw)//2]^=1;payload.write_bytes(raw)
        try:go2_load_checkpoint(bad_dir)
        except RuntimeError:pass
        else:raise AssertionError('Corrupt checkpoint silently restarted')
    report={'schema':'ahr-k19-training-validation-v1','updated_at_utc':go2_now(),'all_passed':True,'source_identity':go2_k19_sources(project),
        'checks':['typed legal-state batches','masked auxiliary objectives','policy-only row weighting','invalid weight refusal','missing economic target refusal',
                  'exact CPU/GPU fresh-kernel recovery','latest-two/best retention','strict corruption/data mismatch refusal'],
        'fixtures':fixtures,'production_training_started':False}
    go2_atomic_json(project/'reports/ahr_v2/k19_training_validation.json',report);return report

### Detached launch and exact resume

Run All launches the registered pair after checking a matching short recovery receipt and a complete competitive cache. Repeated Run All attaches to its status or resumes the same experiment; it does not create a second writer. For the complete workflow, notebook 20's integrated controller can call `go2_k19_train` directly and own the common pipeline lock.

In [ ]:
def go2_k19_pair_configs(pair_id='ahr-k19-20261011',cache_id='go9x9-competitive-v2',device='cuda:1'):
    if not pair_id or Path(pair_id).name!=pair_id:raise ValueError('Simple pair identifier required')
    return [go2_k19_config(pair_id+'-'+variant,cache_id,device,variant) for variant in ('control','challenger')]

def go2_k19_launch(project=None,pair_id='ahr-k19-20261011',cache_id='go9x9-competitive-v2',device='cuda:1'):
    project=go2_project(project);sources=go2_k19_sources(project);receipt_path=project/'reports/ahr_v2/k19_training_validation.json'
    if not receipt_path.exists():raise RuntimeError('Run go2_k19_validate before the long GPU launch')
    receipt=json.loads(receipt_path.read_text())
    if receipt.get('all_passed') is not True or receipt['source_identity']!=sources or not any(row['device']==device and row['exact_scientific_state'] for row in receipt['fixtures']):raise RuntimeError('Recovery receipt does not match current sources/device')
    configs=go2_k19_pair_configs(pair_id,cache_id,device);dataset=go2_k18_dataset(project,cache_id=cache_id,verify=True)
    if dataset['train']['count']<2000000 or dataset['dev']['count']<16384:raise RuntimeError('Default paired experiment requires the registered 2m/16384 competitive bank')
    initial_pins=[go2_k19_initial_player(project,config)[1] for config in configs]
    packet={'schema':'ahr-k19-pair-launch-v1','pair_id':pair_id,'sources':sources,'configs':configs,
            'data_identity':go2_k19_data_identity(dataset),'initialization_pins':initial_pins,'validation_sources':receipt['source_identity'],'validated_devices':sorted(row['device'] for row in receipt['fixtures'])}
    packet_path=project/'config'/('k19-'+pair_id+'.json');packet_path.parent.mkdir(parents=True,exist_ok=True)
    if packet_path.exists():
        if json.loads(packet_path.read_text())!=packet:raise RuntimeError('Existing paired configuration differs; refuse restart')
    else:go2_atomic_json(packet_path,packet)
    folder=project/'reports/ahr_v2'/pair_id;folder.mkdir(parents=True,exist_ok=True);worker=folder/'detached_training_worker.ipynb';status_path=folder/'worker.json'
    session='ahr-k19-'+hashlib.sha256(pair_id.encode()).hexdigest()[:12]
    info={'pair_id':pair_id,'session':session,'worker_notebook':str(worker),'log':str(folder/'tmux.log'),
        'attach':'tmux attach -t '+session,'detach':'Ctrl-b then d','resume':'Run All in notebook 19 with the identical pair ID/cache/device',
        'candidate_run_ids':[c['run_id'] for c in configs]}
    if status_path.exists() and json.loads(status_path.read_text()).get('status')=='COMPLETE':return {**info,'status':'COMPLETE','launched':False}
    if subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0:return {**info,'status':'RUNNING','launched':False}
    bootstrap=go2_k19_bootstrap_text(project)
    runner="PACKET=json.loads((PROJECT/'config'/"+repr(packet_path.name)+").read_text())\nif go2_k19_sources(PROJECT)!=PACKET['sources']:raise RuntimeError('Detached source identity changed')\nPAIR_DIR=PROJECT/'reports/ahr_v2'/PACKET['pair_id']\nwith go2_checkpoint_lock(PROJECT/'checkpoints'/('k19-pair-'+PACKET['pair_id'])):\n    go2_atomic_json(PAIR_DIR/'worker.json',{'status':'RUNNING','started_utc':go2_now()})\n    try:\n        for config in PACKET['configs']:\n            state=go2_k19_train(PROJECT,config)\n            go2_atomic_json(PAIR_DIR/'worker.json',{'status':'RUNNING','last_candidate':config['run_id'],'last_step':state['global_step'],'updated_utc':go2_now()})\n            if state['status'] in ('STOPPED','PAUSED'):raise RuntimeError('Candidate stopped before paired experiment completed')\n        go2_atomic_json(PAIR_DIR/'worker.json',{'status':'COMPLETE','completed_utc':go2_now(),'candidate_run_ids':[c['run_id'] for c in PACKET['configs']]})\n    except BaseException as error:\n        go2_atomic_json(PAIR_DIR/'worker.json',{'status':'ERROR','error':repr(error),'updated_utc':go2_now()})\n        raise"
    notebook=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('Detached registered Scout/Worker pair. Rerun this worker to restore the exact existing states.'),
        nbformat.v4.new_code_cell(bootstrap),nbformat.v4.new_code_cell(runner)],metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(notebook,worker)
    command=shlex.join([str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
                      '--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)])+' > '+shlex.quote(str(folder/'tmux.log'))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],check=True)
    go2_atomic_json(folder/'launch.json',{**info,'packet_sha256':go2_file_sha(packet_path),'launched_utc':go2_now()})
    return {**info,'status':'RUNNING','launched':True}

def go2_k19_status(project=None,pair_id='ahr-k19-20261011'):
    project=Path(project or PROJECT);rows=[]
    for config in go2_k19_pair_configs(pair_id):
        path=project/'reports/ahr_v2'/config['run_id']/'status.json'
        if path.exists():
            status=json.loads(path.read_text());audit=status.get('last_development') or {};all_rows=audit.get('groups',{}).get('all',{})
            rows.append({'Candidate':config['variant'],'Status':status['status'],'Updates':status['global_step'],'Rows seen':status['rows_seen'],
                'Active training (min)':round(status['active_training_seconds']/60,2),'Dev policy KL':all_rows.get('policy_kl'),
                'Dev value RMSE':math.sqrt(all_rows['teacher_value_mse']) if 'teacher_value_mse' in all_rows else None})
        else:rows.append({'Candidate':config['variant'],'Status':'NOT_STARTED','Updates':None,'Rows seen':None,'Active training (min)':None,'Dev policy KL':None,'Dev value RMSE':None})
    return rows

def go2_k19_show(project=None,pair_id='ahr-k19-20261011',validation=None):
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display,Markdown
    project=Path(project or PROJECT);display(pd.DataFrame(go2_k19_status(project,pair_id)))
    histories={}
    for variant in ('control','challenger'):
        path=project/'reports/ahr_v2'/(pair_id+'-'+variant)/'development.jsonl'
        if path.exists():histories[variant]=[json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if histories:
        plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
        fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained');colors={'control':'#205493','challenger':'#a56019'}
        for variant,rows in histories.items():
            axes[0].plot([r['optimizer_step'] for r in rows],[r['groups']['all']['policy_kl'] for r in rows],label=variant,color=colors[variant],marker='o',markersize=3)
            subset=[r for r in rows if 'competitive_abs_value_below_095' in r['groups']]
            axes[1].plot([r['optimizer_step'] for r in subset],[math.sqrt(r['groups']['competitive_abs_value_below_095']['teacher_value_mse']) for r in subset],label=variant,color=colors[variant],marker='o',markersize=3)
        axes[0].set(title='Scout error on the fixed development sample',xlabel='Optimizer updates',ylabel='Teacher-policy KL (nats)')
        axes[1].set(title='Worker error on competitive development positions',xlabel='Optimizer updates',ylabel='Signed teacher-value RMSE')
        for ax in axes:ax.grid(axis='y',alpha=.2);ax.legend()
        plt.show();display(Markdown('Development predictive metrics support candidate review. Fresh match results in notebook 20 determine playing promotion.'))
    elif validation:
        display(pd.DataFrame([{k:fixture[k] for k in ('device','precision','steps','rows_seen','epoch','cursor','exact_scientific_state')} for fixture in validation['fixtures']]))
        display(Markdown('**Short recovery checks passed. Production training is not started.** The table above describes disposable engineering fixtures, not Go strength.'))
    else:display(Markdown('**Ready for the competitive bank from notebook 18.** No production training measurements exist yet.'))

### Run settings

Keep the same IDs for exact resume. The normal mode launches training after preflight; validation mode runs only disposable six-update fixtures. The final cell reports the tmux session and checkpoint run IDs. If the integrated notebook-20 pipeline is already running, use this notebook to view status; do not launch a second pair.

In [ ]:
K19_MODE=os.environ.get('AHR_K19_MODE','train')
K19_PAIR_ID='ahr-k19-20261011'
K19_CACHE_ID='go9x9-competitive-v2'
K19_DEVICE='cuda:1'
if K19_MODE not in ('train','validate','status'):raise ValueError('AHR_K19_MODE must be train, validate, or status')
K19_VALIDATION=None
if K19_MODE=='validate':
    K19_VALIDATION=go2_k19_validate(PROJECT)
elif K19_MODE=='train':
    receipt=PROJECT/'reports/ahr_v2/k19_training_validation.json'
    if not receipt.exists() or json.loads(receipt.read_text()).get('source_identity')!=go2_k19_sources(PROJECT):
        K19_VALIDATION=go2_k19_validate(PROJECT)
    K19_LAUNCH=go2_k19_launch(PROJECT,K19_PAIR_ID,K19_CACHE_ID,K19_DEVICE)
    from IPython.display import display,Markdown
    display(Markdown('**'+K19_LAUNCH['status']+'** — tmux session `'+K19_LAUNCH['session']+'`. Attach with `'+K19_LAUNCH['attach']+'`; detach with Ctrl-b then d.'))
go2_k19_show(PROJECT,K19_PAIR_ID,K19_VALIDATION)

### Resume and next decision

A stopped or disconnected run resumes from the latest verified checkpoint using the saved configuration. For one candidate: `saved_config=json.loads((PROJECT/'checkpoints'/run_id/'config.json').read_text())`, then `go2_k19_train(PROJECT,saved_config)`. Remove an intentionally created `STOP` marker only when you intend to resume that run. Invalid manifests or changed sources/data/configurations raise an error and require investigation.

Compare the fixed development sample by phase and competitive value, then ask notebook 20 to run fresh paired-color KataGo games at the actual AHR move-time budget. A lower supervised loss is not a promise of beating KataGo, and local AHR proofs remain bounded tactical facts.